In [0]:
%run ./helper

In [0]:
from pyspark.sql.functions import col, current_timestamp, lit, from_json

dbutils.widgets.dropdown("source_catalog", catalog_list[0], catalog_list)
dbutils.widgets.dropdown("source_schema", schema_list[0], schema_list)
catalog=dbutils.widgets.get("source_catalog")
schema=dbutils.widgets.get("source_schema")

evaluation_table_name=f"{catalog}.{schema}.model_evaluation"
topic_info_table_name=f"{catalog}.{schema}.topic_info_local"
queue_table_name=f"{catalog}.{schema}.{refinement_queue_table_name}"
queue_checkpoint_path=f"/Volumes/{catalog}/{schema}/checkpoints/{refinement_queue_table_name}_model_evaluation.txt"


In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {queue_table_name} (
  execution_id STRING,
  model STRING,
  layer INT,
  topic STRING,
  count BIGINT,
  score INT,
  description STRING,
  status STRING,
  created_at TIMESTAMP,
  started_at TIMESTAMP,
  finished_at TIMESTAMP,
  error_message STRING
)
USING DELTA
TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")
print(f"queue_table_name:{queue_table_name}")


In [0]:
from datetime import datetime, timedelta
last_version = read_checkpoint(queue_checkpoint_path)
current_version = get_current_version(evaluation_table_name)

if current_version is None:
    dbutils.notebook.exit(f"No versions found for {evaluation_table_name}")

if last_version is None:
    evaluation_changes_df = spark.table(evaluation_table_name)
elif last_version >= current_version:
    dbutils.notebook.exit(
        f"No new evaluation rows to enqueue. Checkpoint={last_version}, latest={current_version}"
    )
else:
    try:
        evaluation_changes_df = spark.sql(f"""
            SELECT execution_id, model, layer, topic, ai_grade
            FROM table_changes('{evaluation_table_name}', {last_version + 1}, {current_version})
            WHERE _change_type = 'insert'
        """)
    except Exception as e:
        msg = str(e).lower()
        
        # Check if the error is due to Delta retention limits
        if "delta_unsupported_time_travel" in msg or "deletedfileretentionduration" in msg:
            # Calculate the timestamp for 24 hours ago
            fallback_start_time = (datetime.now() - timedelta(days=1)).strftime("%Y-%m-%d %H:%M:%S")
            
            print(f"Warning: CDF retention expired for version {last_version + 1}. "
                f"Falling back to CDF for the last 24 hours (starting {fallback_start_time}).")
            
            # FIX: Omit the end version. Providing just the start timestamp automatically reads to the latest state.
            evaluation_changes_df = spark.sql(f"""
                SELECT execution_id, model, layer, topic, ai_grade
                FROM table_changes('{evaluation_table_name}', '{fallback_start_time}')
                WHERE _change_type = 'insert'
            """)
            
        else:
            # If it's a completely different error, raise it so you don't swallow bugs
            raise e


In [0]:
evaluation_with_grade_df = evaluation_changes_df.withColumn(
    "ai_grade_parsed",
    from_json(col("ai_grade"), "score INT, description STRING")
)

eligible_df = (
    evaluation_with_grade_df.alias("e")
    .join(
        spark.table(topic_info_table_name).alias("t"),
        on=["execution_id", "model", "layer", "topic"],
        how="inner"
    )
    .filter(
        (col("layer") == 0)
        & (col("t.count") > refinement_min_topic_count)
        & (col("e.ai_grade_parsed").isNotNull())
        & (col("e.ai_grade_parsed.score") < refinement_score_threshold)
    )
    .select(
        col("execution_id"),
        col("model"),
        col("layer"),
        col("topic").cast("string").alias("topic"),
        col("t.count").cast("bigint").alias("count"),
        col("e.ai_grade_parsed.score").cast("int").alias("score"),
        col("e.ai_grade_parsed.description").cast("string").alias("description"),
        lit("pending").alias("status"),
        current_timestamp().alias("created_at"),
        lit(None).cast("timestamp").alias("started_at"),
        lit(None).cast("timestamp").alias("finished_at"),
        lit(None).cast("string").alias("error_message")
    )
    .dropDuplicates(["execution_id", "model", "layer", "topic"])
)

if eligible_df.limit(1).count() == 0:
    write_checkpoint(queue_checkpoint_path, current_version)
    dbutils.notebook.exit(
        f"No eligible low-score layer-0 topics found. Checkpoint updated to {current_version}"
    )


In [0]:
eligible_df.createOrReplaceTempView("eligible_refinement_topics")

merge_sql = f"""
MERGE INTO {queue_table_name} AS target
USING eligible_refinement_topics AS source
ON target.execution_id = source.execution_id
   AND target.model = source.model
   AND target.layer = source.layer
   AND target.topic = source.topic
WHEN NOT MATCHED THEN
  INSERT (
    execution_id,
    model,
    layer,
    topic,
    count,
    score,
    description,
    status,
    created_at,
    started_at,
    finished_at,
    error_message
  )
  VALUES (
    source.execution_id,
    source.model,
    source.layer,
    source.topic,
    source.count,
    source.score,
    source.description,
    source.status,
    source.created_at,
    source.started_at,
    source.finished_at,
    source.error_message
  )
"""
spark.sql(merge_sql)
write_checkpoint(queue_checkpoint_path, current_version)
display(spark.table(queue_table_name).orderBy(col("created_at").desc()).limit(20))
